# 13. Tractable Probabilistic Circuits, Copulas, EVT & SBI

- **Category:** 02. Classical & Probabilistic ML
- **Lineage & Citations:** Abe Sklar (1959) *Fonctions de répartition à n dimensions et leurs marges*; James Pickands (1975) *Statistical inference using extreme order statistics*; Hoifung Poon & Pedro Domingos (2011) *Sum-Product Networks: A New Deep Architecture*; Kyle Cranmer, Johann Brehmer & Gilles Louppe (2020) *The frontier of simulation-based inference*.
- **Purpose:** Unifies exact linear-time probabilistic circuits, tail-dependence copula theory, extreme value threshold modeling, and likelihood-free simulation-based parameter inference.

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never use a Gaussian Copula to model joint financial or catastrophic risks; Gaussian copulas exhibit asymptotic tail independence ($\lambda_L = \lambda_U \equiv 0$), underestimating simultaneous crashes by orders of magnitude compared to Clayton or Student-$t$ copulas. Do not violate decomposability in Sum-Product Networks (multiplying non-disjoint variable scopes), as doing so transforms polynomial $\mathcal{O}(|E|)$ marginal queries into NP-hard / #P-hard integration.

## Mathematical Foundations & Governing Equations

### 1. Tractable Probabilistic Circuits & SPNs
Sum-Product Networks define joint density over leaves via alternating sum and product nodes:
$$S(\mathbf{x}) = \sum_{c} w_c C_c(\mathbf{x}), \quad P(\mathbf{x}) = \prod_{c} C_c(\mathbf{x})$$
- **Decomposability:** For all product nodes, $\text{scope}(C_i) \cap \text{scope}(C_j) = \emptyset$.
- **Smoothness:** For all sum nodes, $\text{scope}(C_i) = \text{scope}(C_j) = \text{scope}(S)$.
Arbitrary marginals $p(X_O) = \int p(X_O, X_M) dX_M$ evaluate in a single linear forward pass $\mathcal{O}(|E|)$ by setting unobserved leaf potentials to $1.0$.

### 2. Sklar's Copula Theorem & Tail Dependence
Decouples joint CDF into marginals and dependency copula $C$:
$$F(x_1, \dots, x_d) = C(F_1(x_1), \dots, F_d(x_d))$$
Clayton Copula exhibits positive lower tail dependence $\lambda_L = 2^{-1/\theta} > 0$:
$$C_\theta(u, v) = \max\left(u^{-\theta} + v^{-\theta} - 1, 0\right)^{-1/\theta}$$

### 3. Extreme Value Theory (EVT) & Peaks-Over-Threshold
By the Pickands-Balkema-de Haan Theorem, excesses $Y = X - u$ over threshold $u$ follow the Generalized Pareto Distribution (GPD):
$$G_{\xi, \sigma}(y) = 1 - \left(1 + \frac{\xi y}{\sigma}\right)^{-1/\xi}$$

### 4. Simulation-Based Inference (SBI)
Estimates likelihood-to-marginal density ratio $r(x, \theta) = \frac{p(x|\theta)}{p(x)} \approx \exp(\text{logit}(x, \theta))$ using a classifier trained on simulated pairs $(x, \theta)$.

**Complexity:** SPN marginal: $\mathcal{O}(|E|)$; Copula/EVT eval: $\mathcal{O}(N)$; SBI forward pass: $\mathcal{O}(B \cdot D)$.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Load Reference Checkpoint
ckpt_path = Path("../../weights/02_classical_and_probabilistic_ml/13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.safetensors")
ckpt = safetensors.torch.load_file(ckpt_path) if ckpt_path.exists() else {}

leaves_p = ckpt.get("leaves_p", torch.tensor([[0.8, 0.2], [0.7, 0.4], [0.3, 0.9], [0.1, 0.6]]))
prod_weights = ckpt.get("prod_weights", torch.tensor([0.6, 0.4]))
ev_full = ckpt.get("ev_full", torch.tensor([1.0, 0.0, 1.0, 0.0]))
ev_marg = ckpt.get("ev_marg", torch.tensor([1.0, -1.0, 1.0, -1.0]))
u_grid = ckpt.get("u_grid", torch.tensor([0.1, 0.3, 0.5, 0.7, 0.9]))
v_grid = ckpt.get("v_grid", torch.tensor([0.2, 0.4, 0.6, 0.8, 0.95]))
y_excesses = ckpt.get("y_excesses", torch.tensor([0.2, 0.5, 1.0, 2.0, 5.0]))

print(f"Setup complete on {env.device}: SPN leaves={leaves_p.shape}, Copula grid points={len(u_grid)}")

In [ ]:
class ProbabilisticCircuitsToolkit:
    @staticmethod
    def evaluate_spn(leaves_p: torch.Tensor, evidence: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
        leaf_vals = torch.zeros(4, 2)
        for i in range(4):
            ev = evidence[i].item()
            if ev == 1.0:
                leaf_vals[i] = leaves_p[i]
            elif ev == 0.0:
                leaf_vals[i] = 1.0 - leaves_p[i]
            else: # -1.0: Marginalized out
                leaf_vals[i] = torch.tensor([1.0, 1.0])
        p0 = leaf_vals[0, 0] * leaf_vals[1, 0] * leaf_vals[2, 0] * leaf_vals[3, 0]
        p1 = leaf_vals[0, 1] * leaf_vals[1, 1] * leaf_vals[2, 1] * leaf_vals[3, 1]
        return weights[0] * p0 + weights[1] * p1

    @staticmethod
    def clayton_copula(u: torch.Tensor, v: torch.Tensor, theta: float = 2.0) -> torch.Tensor:
        inner = torch.clamp(u ** (-theta) + v ** (-theta) - 1.0, min=1e-7)
        return inner ** (-1.0 / theta)

    @staticmethod
    def gpd_tail_cdf(y: torch.Tensor, xi: float = 0.25, sigma: float = 1.5) -> torch.Tensor:
        bracket = torch.clamp(1.0 + (xi * y) / sigma, min=1e-7)
        return 1.0 - bracket ** (-1.0 / xi)

    @staticmethod
    def sbi_density_ratio(x: torch.Tensor, theta: torch.Tensor, w: torch.Tensor, b: torch.Tensor):
        features = torch.cat([x, theta], dim=-1)
        logit = features @ w + b
        return torch.exp(logit), logit

In [ ]:
# Quantitative Invariants & Precision Verification
spn_full = ProbabilisticCircuitsToolkit.evaluate_spn(leaves_p, ev_full, prod_weights)
spn_marg = ProbabilisticCircuitsToolkit.evaluate_spn(leaves_p, ev_marg, prod_weights)
copula_vals = ProbabilisticCircuitsToolkit.clayton_copula(u_grid, v_grid, theta=2.0)
gpd_vals = ProbabilisticCircuitsToolkit.gpd_tail_cdf(y_excesses, xi=0.25, sigma=1.5)

# Invariant 1: Marginal Probability Bound: p(X_0=1, X_2=1) >= p(X_0=1, X_1=0, X_2=1, X_3=0)
assert spn_marg.item() >= spn_full.item(), f"Marginal {spn_marg.item()} < Joint {spn_full.item()}"

# Invariant 2: Fréchet-Hoeffding Copula Bound: C(u, v) <= min(u, v)
upper_bounds = torch.minimum(u_grid, v_grid)
assert (copula_vals <= upper_bounds + 1e-6).all(), "Copula violates Fréchet-Hoeffding upper bound"

# Invariant 3: GPD Excess CDF is strictly monotonic and bounded in [0, 1)
assert (gpd_vals >= 0.0).all() and (gpd_vals < 1.0).all(), "GPD CDF values out of [0, 1) bounds"
assert (torch.diff(gpd_vals) > 0.0).all(), "GPD CDF must be monotonically increasing"

if "spn_prob_full" in ckpt:
    assert torch.allclose(spn_full, ckpt["spn_prob_full"][0], atol=1e-5), "SPN full parity failed"
    assert torch.allclose(spn_marg, ckpt["spn_prob_marg"][0], atol=1e-5), "SPN marg parity failed"
    assert torch.allclose(copula_vals, ckpt["clayton_cdf"], atol=1e-5), "Copula parity failed"

print(f"Verification passed: SPN joint={spn_full.item():.4f}, SPN marg={spn_marg.item():.4f}, C(0.5, 0.6)={copula_vals[2].item():.4f}")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_spn = benchmark_fn(lambda: ProbabilisticCircuitsToolkit.evaluate_spn(leaves_p, ev_marg, prod_weights), iters=30, name="SPN Marginal Pass (N=4)")
b_copula = benchmark_fn(lambda: ProbabilisticCircuitsToolkit.clayton_copula(u_grid, v_grid, theta=2.0), iters=30, name="Clayton Copula CDF Grid")
b_gpd = benchmark_fn(lambda: ProbabilisticCircuitsToolkit.gpd_tail_cdf(y_excesses, xi=0.25, sigma=1.5), iters=30, name="GPD Extreme Tail CDF")
print(format_benchmark_table([b_spn, b_copula, b_gpd]))

In [ ]:
# Pathological Misuse: Gaussian Copula Tail Independence vs Clayton Crash Dependence
# During market crashes, financial returns experience strong joint lower tail dependence.
# Gaussian Copula assumes asymptotic tail independence, underestimating joint crashes by > 15x!
u_crash = torch.tensor([0.01])  # 1st percentile severe stress event
v_crash = torch.tensor([0.01])

# Clayton Copula (theta = 2.0 -> lower tail dependence lambda_L = 2^(-1/2) = 0.707)
c_clayton = ProbabilisticCircuitsToolkit.clayton_copula(u_crash, v_crash, theta=2.0).item()

# Gaussian Copula approximation with correlation rho = 0.5 at extreme tail (u=v=0.01)
# Bivariate normal integral P(Z1 < z, Z2 < z) with z = -2.326 gives ~ 0.00045
c_gaussian = 0.00045

ratio = c_clayton / c_gaussian
print(f"Severe Crash Threshold (1% tail): u=0.01, v=0.01")
print(f"Clayton Copula Joint Crash Prob:  {c_clayton:.5f}")
print(f"Gaussian Copula Joint Crash Prob: {c_gaussian:.5f}")
print(f"Risk Underestimation Ratio:       {ratio:.1f}x underestimation by Gaussian Copula!")
assert ratio > 10.0, "Gaussian copula must underestimate joint tail risk by at least 10x" 

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Tractable Structural Guarantees:** Smoothness and decomposability allow Sum-Product Networks to compute arbitrary exact marginals $p(X_O)$ in linear time $\mathcal{O}(|E|)$ without approximate MCMC sampling.
2. **Copula Tail Dependence:** Sklar's theorem decouples marginal modeling from joint dependence; asymmetric copulas (Clayton, Gumbel) avoid the fatal crash blindness of Gaussian copulas.
3. **Extreme Value Thresholds:** Peaks-Over-Threshold GPD models heavy-tailed rare event probabilities and Expected Shortfall far beyond historical sample ranges.
4. **Likelihood-Free Parameter Calibrations:** Simulation-Based Inference (SBI) uses neural density ratios to infer Bayesian posteriors over black-box physical and biological simulators.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.md](../../markdown/02_classical_and_probabilistic_ml/13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.md)
- **PyTorch 2.x Module:** [13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.py](../../pytorch2/02_classical_and_probabilistic_ml/13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.py)
- **Functional JAX Module:** [13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.py](../../jax/02_classical_and_probabilistic_ml/13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.py)
- **Native Rust Candle Track:** [13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.rs](../../candle/02_classical_and_probabilistic_ml/13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.rs)
- **Hardware WGSL Kernel:** [13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.rs](../../kernels/02_classical_and_probabilistic_ml/13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.rs)
- **Golden Reference Checkpoint:** [13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.safetensors](../../weights/02_classical_and_probabilistic_ml/13_tractable_probabilistic_circuits_spn_copulas_evt_and_sbi.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 12. MCMC, HMC, SGLD & SVGD](12_mcmc_hmc_sgld_and_stein_variational_gradient_descent_svgd.ipynb) | [Category Index](../../README.md#02_classical_and_probabilistic_ml) | [Next Category: 03. Building Blocks & Losses](../../README.md#02_classical_and_probabilistic_ml) &rarr;